In [8]:
import polars as pl
import numpy as np
from scipy import stats
from pathlib import Path

In [9]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/18_Pairwise Ranking HepG2 REMC/')
DATASET_PATH = WORKING_PATH / 'dataset'
OUTPUT_PATH  = WORKING_PATH / 'output' / 'combined'

In [10]:
# One sample test
def one_sample_test_against_chance(accuracies, p_null=0.5):
    accuracies = np.asarray(accuracies, dtype=float)
    n = len(accuracies)
    mean = accuracies.mean()
    sd = accuracies.std(ddof=1)
    se = sd / np.sqrt(n)
 
    # one-sample t-test -- preferred for small n (e.g. n=5 runs)
    t_stat, t_p = stats.ttest_1samp(accuracies, popmean=p_null)
 
    # z-test shown for reference; not appropriate for n this small
    z_stat = (mean - p_null) / se
    z_p = 2 * (1 - stats.norm.cdf(abs(z_stat)))
 
    return {
        "n_runs": n,
        "mean_accuracy": mean,
        "sd": sd,
        "t_stat": t_stat,
        "t_p_value": t_p,
        "z_stat": z_stat,
        "z_p_value": z_p,
    }

In [11]:
# Pretty report
def report(name, accuracies):
    r = one_sample_test_against_chance(accuracies)
    print(f"{name}:")
    print(f"  mean = {r['mean_accuracy']*100:.2f}% +/- {r['sd']*100:.2f}% "
          f"(n={r['n_runs']} runs)")
    print(f"  t-test:  t({r['n_runs']-1}) = {r['t_stat']:.3f}, "
          f"p = {r['t_p_value']:.4g}")
    print(f"  z-test:  z = {r['z_stat']:.3f}, p = {r['z_p_value']:.4g}")
    print()

In [12]:
### Merge the results into single CSV file

# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-test-metrics.csv")

In [13]:
pl_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K4me3""",12,75.4,0.84,74.9,0.8437,0.7218,0.7745,0.7472,0.922
"""LogisticRegression""",1011,"""H3K4me3""",null,75.2,0.8226,76.6,0.8425,0.7966,0.6868,0.7377,0.797
"""RandomForest""",1011,"""H3K4me3""",null,75.9,0.8281,77.2,0.8531,0.7898,0.714,0.75,0.788
"""SVM_Linear""",1011,"""H3K4me3""",null,75.1,0.8231,76.8,0.8428,0.7976,0.691,0.7405,0.794
"""DirectRanker""",123,"""H3K4me3""",14,77.1,0.86,73.0,0.8044,0.7097,0.7857,0.7458,0.922
…,…,…,…,…,…,…,…,…,…,…,…
"""SVM_Linear""",456,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,75.2,0.8224,75.9,0.8204,0.7665,0.7337,0.7497,0.759
"""DirectRanker""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",54,79.2,0.87,76.4,0.8592,0.7436,0.7835,0.7631,0.989
"""LogisticRegression""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,73.4,0.8155,72.9,0.8125,0.7238,0.7134,0.7186,0.777


# Chek for the H3K9me3 results

In [14]:
# Check for the H3K9me3
H3K9me3_DirectRanker_df = pl_df.filter((pl.col("model") == "DirectRanker") & (pl.col("histone_marker") == "H3K9me3"))
H3K9me3_DirectRanker_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K9me3""",100,46.8,0.53,52.3,0.5593,0.5011,0.9582,0.6581,0.178
"""DirectRanker""",123,"""H3K9me3""",100,51.6,0.52,50.9,0.5252,0.5072,0.9127,0.652,0.175
"""DirectRanker""",42,"""H3K9me3""",100,49.0,0.53,52.7,0.5555,0.5098,0.9532,0.6643,0.161
"""DirectRanker""",456,"""H3K9me3""",100,51.2,0.57,51.2,0.5372,0.5022,0.9289,0.6519,0.16
"""DirectRanker""",789,"""H3K9me3""",100,48.7,0.53,52.4,0.5499,0.5049,0.9629,0.6624,0.155


In [15]:
# Select the accuracy
test_accuracies = H3K9me3_DirectRanker_df["test_accuracy"].to_list()
test_accuracies = np.array(test_accuracies) / 100
test_accuracies

array([0.523, 0.509, 0.527, 0.512, 0.524])

In [16]:
report("H3K9me3", test_accuracies)

H3K9me3:
  mean = 51.90% +/- 0.80% (n=5 runs)
  t-test:  t(4) = 5.332, p = 0.005959
  z-test:  z = 5.332, p = 9.739e-08



# Check for H3K27me3

In [17]:
# Check for the H3K27me3
H3K27me3_DirectRanker_df = pl_df.filter((pl.col("model") == "DirectRanker") & (pl.col("histone_marker") == "H3K27me3"))
H3K27me3_DirectRanker_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K27me3""",100,64.9,0.7,68.7,0.7501,0.6566,0.7265,0.6898,0.933
"""DirectRanker""",123,"""H3K27me3""",100,66.6,0.72,64.9,0.7077,0.6446,0.6766,0.6602,0.943
"""DirectRanker""",42,"""H3K27me3""",100,64.5,0.7,62.1,0.6892,0.6011,0.6782,0.6373,0.951
"""DirectRanker""",456,"""H3K27me3""",100,67.0,0.75,65.5,0.7187,0.6395,0.685,0.6614,0.944
"""DirectRanker""",789,"""H3K27me3""",100,65.3,0.72,63.8,0.7017,0.6185,0.6619,0.6394,0.948


In [18]:
# Select the accuracy
test_accuracies = H3K27me3_DirectRanker_df["test_accuracy"].to_list()
test_accuracies = np.array(test_accuracies) / 100
test_accuracies

array([0.687, 0.649, 0.621, 0.655, 0.638])

In [19]:
report("H3K27me3", test_accuracies)

H3K27me3:
  mean = 65.00% +/- 2.44% (n=5 runs)
  t-test:  t(4) = 13.750, p = 0.0001621
  z-test:  z = 13.750, p = 0

